# Session 32b — Embeddings & Vector Databases
### A step-by-step teaching notebook

**How to use this notebook.** It is written for *you*, the instructor, to work through
**live** with the client. Every step follows the same rhythm:

| Part | What you do |
|---|---|
| 🎯 **What** | One sentence: the thing we are about to build |
| 💡 **Why** | The problem it solves — say this *before* typing |
| ⌨️ **Code** | Type it live. Only this step's code, nothing more |
| 🔍 **How it works** | Line by line |
| ⚙️ **Behind the scenes** | What actually happens at runtime |
| 🧩 **Pipeline fit** | Where this piece sits in the RAG picture from Session 32a |
| ❓ **Likely questions** | Prepared answers for what they will ask |
| ✅ **Checkpoint** | Pause. Ask. Do not move on until they answer |

**Path through the notebook**

```
Step 0  Setup and corpus
Step 1  One embedding                 ─┐
Step 2  Embed the whole corpus         │  "what is a vector?"
Step 3  Cosine similarity by hand      │
Step 4  Brute-force search            ─┘
Step 5  FAISS — exact index           ─┐
Step 6  FAISS — approximate (IVF)      │  "how do we make it fast?"
Step 7  ChromaDB — a real database     │
Step 8  Pinecone — managed (optional) ─┘
Step 9  Close the loop: a mini RAG answer
Step 10 Recap, comparison, exercises
```

**Before the session**

1. `pip install openai numpy faiss-cpu chromadb tiktoken` (add `pinecone` for Step 8).
2. Put a working `OPENAI_API_KEY` in your environment — run Step 1 once to confirm.
3. Total embedding cost for this notebook: well under one cent.

> **Timing:** Steps 0–4 take about 25 minutes, Steps 5–8 about 30, Step 9 about 10.
> If you are running short, Step 6 and Step 8 are the two you can safely drop.

---

## Step 0 · Setup and the corpus

### 🎯 What we are going to implement


Install the libraries, load the API key, and define a tiny corpus of ten
company-handbook sentences with metadata attached to each one.


### 💡 Why we are implementing it


Everything else in the notebook operates on this corpus. Keeping it small and
readable means the client can *check the answers with their own eyes* — when a search
returns chunk 3, they can see immediately whether chunk 3 was the right answer.
A 10,000-chunk corpus teaches nothing at this stage because nobody can verify it.

Metadata is included from the very first line on purpose: filtering is not an
advanced feature you bolt on later, it is part of the record from the start.

In [ ]:
# Run once per machine. Skip the line for Pinecone if you are not doing Step 8.
!pip install -q openai numpy faiss-cpu chromadb tiktoken
# !pip install -q pinecone

In [1]:
import os, json, time
import numpy as np

# The API key is read from the environment — never paste a key into a notebook
# you might share. In a terminal: export OPENAI_API_KEY="sk-..."
from openai import OpenAI
client = OpenAI()          # picks up OPENAI_API_KEY automatically

EMBED_MODEL = "text-embedding-3-small"     # 1536 dimensions

# --- our miniature knowledge base -------------------------------------------
DOCS = [
    {"id": "hr-1",   "text": "New employees receive 15 days of paid annual leave in their first year, "
                             "increasing to 20 days after two years of service.",
     "source": "handbook.pdf", "section": "4.2", "category": "hr"},
    {"id": "hr-2",   "text": "The company matches 401k contributions up to 4% of salary. "
                             "Vesting is immediate for all employees.",
     "source": "handbook.pdf", "section": "4.7", "category": "hr"},
    {"id": "hr-3",   "text": "Remote work is available up to three days per week with manager approval.",
     "source": "handbook.pdf", "section": "2.1", "category": "hr"},
    {"id": "pol-1",  "text": "Refund requests must be submitted within 30 days of purchase. "
                             "After 30 days only store credit is available.",
     "source": "policies.md", "section": "1.1", "category": "policy"},
    {"id": "pol-2",  "text": "Enterprise customers receive dedicated support with a four-hour "
                             "response time SLA. Standard support responds within 24 hours.",
     "source": "policies.md", "section": "1.4", "category": "policy"},
    {"id": "eng-1",  "text": "Production deployments require approval from the team lead and a "
                             "passing CI pipeline. Hotfixes may bypass lead approval.",
     "source": "runbook.md", "section": "3.2", "category": "engineering"},
    {"id": "eng-2",  "text": "The API rate limit is 100 requests per minute on the Free tier and "
                             "1000 requests per minute on the Pro tier.",
     "source": "api-docs.md", "section": "9.1", "category": "engineering"},
    {"id": "eng-3",  "text": "Data is encrypted at rest using AES-256 and in transit using TLS 1.3. "
                             "SOC 2 Type II compliance is maintained annually.",
     "source": "security.md", "section": "6.0", "category": "engineering"},
    {"id": "fin-1",  "text": "Vendor invoices above $10,000 require VP approval. Below that amount "
                             "manager approval is sufficient.",
     "source": "finance.md", "section": "5.3", "category": "finance"},
    {"id": "fin-2",  "text": "Expense reports must be filed within 60 days of the expense date.",
     "source": "finance.md", "section": "5.9", "category": "finance"},
]

TEXTS = [d["text"] for d in DOCS]
print(f"{len(DOCS)} chunks loaded")
print(TEXTS[0][:80], "...")

10 chunks loaded
New employees receive 15 days of paid annual leave in their first year, increasi ...


In [2]:
TEXTS

['New employees receive 15 days of paid annual leave in their first year, increasing to 20 days after two years of service.',
 'The company matches 401k contributions up to 4% of salary. Vesting is immediate for all employees.',
 'Remote work is available up to three days per week with manager approval.',
 'Refund requests must be submitted within 30 days of purchase. After 30 days only store credit is available.',
 'Enterprise customers receive dedicated support with a four-hour response time SLA. Standard support responds within 24 hours.',
 'Production deployments require approval from the team lead and a passing CI pipeline. Hotfixes may bypass lead approval.',
 'The API rate limit is 100 requests per minute on the Free tier and 1000 requests per minute on the Pro tier.',
 'Data is encrypted at rest using AES-256 and in transit using TLS 1.3. SOC 2 Type II compliance is maintained annually.',
 'Vendor invoices above $10,000 require VP approval. Below that amount manager approval is

### 🔍 How the code works


- `OpenAI()` with no arguments reads `OPENAI_API_KEY` from the environment. If the key is
  missing you get an `OpenAIError` on the **first API call**, not on this line.
- `DOCS` is a list of dictionaries, not bare strings. Each record already has the four
  fields a real vector-database record has: an **id**, the **text**, and **metadata**
  (`source`, `section`, `category`).
- `TEXTS` is just a convenience view — the plain strings, in the same order as `DOCS`.
  That shared ordering is what lets us map a search result back to its record later.


### ⚙️ Behind the scenes


Nothing has happened over the network yet. This cell only puts Python objects in memory.

Notice what these ten records represent: they are the *output* of the chunking step from
Session 32a. In a real project a loader would have read `handbook.pdf`, a splitter would
have cut it into ~500-token passages, and each passage would have arrived here looking
exactly like one of these dictionaries. We are starting one step downstream so the whole
lesson fits in an hour.


### 🧩 Where this sits in the RAG pipeline


This is the tail end of **Phase 1, step 2 (Chunk)** in the pipeline diagram. Everything
we build from here — embedding, indexing, retrieval — consumes records in this shape.


### ❓ Questions the client is likely to ask

> **“Why keep the text if we are storing vectors?”**  
> Because the vector is only useful for *finding* the chunk. What goes into the LLM prompt is the original text. A vector cannot be read back into words.

> **“Why does every chunk need an id?”**  
> So you can update or delete one record later without rebuilding the whole index — and so a search result can be traced back to a document and page.

> **“How big should a real chunk be?”**  
> Roughly 300–800 tokens with a little overlap. Too small and the passage loses the context that makes it answerable; too big and one chunk covers several topics, which blurs its vector. We tune this properly in Session 33.

### ✅ Checkpoint — pause here


Ask: **“If we later delete `hr-1` from the handbook, what has to happen in our system?”**

*Expected answer:* delete the record (vector + text + metadata) from the store. No retraining,
no re-embedding of anything else. That single property is why RAG beats fine-tuning for
knowledge that changes.


---

## Step 1 · Your first embedding

### 🎯 What we are going to implement


Send one sentence to the OpenAI embeddings endpoint and look carefully at what
comes back.


### 💡 Why we are implementing it


The word “embedding” stays abstract until the client sees the actual numbers. One API
call, printed and dissected, removes most of the mystery: it is a list of floats, it has a
fixed length, and the same text always produces the same list.

In [3]:
response = client.embeddings.create(
    model=EMBED_MODEL,
    input="How many days of paid leave do I get?"
)

vec = response.data[0].embedding          # a plain Python list of floats

print("type          :", type(vec))
print("dimensions    :", len(vec))
print("first 8 values:", [round(v, 4) for v in vec[:8]])
print("tokens billed :", response.usage.total_tokens)
print("model         :", response.model)

type          : <class 'list'>
dimensions    : 1536
first 8 values: [-0.0157, 0.0051, 0.0406, 0.0337, -0.0251, 0.0191, 0.0116, -0.0116]
tokens billed : 10
model         : text-embedding-3-small


### 🔍 How the code works


- `client.embeddings.create(...)` is one HTTPS request. `input` accepts a string **or a
  list of strings** — we use the list form in Step 2.
- The response is an object, not a list: `response.data` is a list of results (one per
  input), and each result carries `.embedding`, the actual vector.
- `response.usage.total_tokens` tells you exactly what you paid for. Embeddings are billed
  per input token; there are no output tokens.
- We round only for printing. **Never round the values you store** — you would be throwing
  away precision that the similarity calculation depends on.


### ⚙️ Behind the scenes


1. Your text is tokenised (roughly 4 characters per token in English).
2. The tokens run through a transformer, producing one contextual vector per token.
3. Those are **pooled** into a single vector for the whole input.
4. OpenAI returns it normalised to unit length — a detail that will matter in Step 3.

The model is a pure function: the same string returns the same vector every time,
and it does not learn anything from your input. Your text is not added to any index
on OpenAI's side.

Cost check: `text-embedding-3-small` gives you roughly **62,500 pages of text per US
dollar** (at ~800 tokens per page). Embedding is the cheap part of a RAG system.


### 🧩 Where this sits in the RAG pipeline


This is the **Embed** box in the pipeline — the one that appears in *both* phases.
Phase 1 calls it once per chunk; Phase 2 calls it once per question.


### ❓ Questions the client is likely to ask

> **“Why 1536 numbers? Why not 10, or a million?”**  
> It is the model's design. More dimensions can hold more nuance but cost more memory and compute. `text-embedding-3-large` uses 3072; you can also ask either model for fewer dimensions with the `dimensions` parameter.

> **“Can I read the numbers? Does index 42 mean 'money'?”**  
> No. Meaning is distributed across all the dimensions. Individual positions are not interpretable — only whole-vector comparisons are.

> **“Does a longer sentence give a longer vector?”**  
> No — always 1536, whether you embed one word or 8,000 tokens. Anything beyond the model's 8,192-token limit is an error, not a truncation.

> **“Is my data used for training?”**  
> API data is not used to train OpenAI models by default. For regulated data, this is the moment to discuss a local embedding model instead.

### ✅ Checkpoint — pause here


Ask: **“If I embed the same sentence twice, do I get the same vector?”**

*Expected answer:* yes — it is a deterministic function. Follow up with: *“And if I embed
it with a different model?”* → a completely different vector, which is why you can never
mix models between indexing and querying.


---

## Step 2 · Embed the whole corpus

### 🎯 What we are going to implement


Embed all ten chunks in a single batched request and store the result as one
NumPy matrix.


### 💡 Why we are implementing it


Two lessons hide in this small cell. First, **batching**: one request with ten inputs is
far faster and cheaper in overhead than ten requests. Second, **the matrix layout** —
every vector store on earth, from a NumPy array to Pinecone, is fundamentally an
`(n_records × n_dimensions)` block of floats plus some bookkeeping.

In [4]:
def embed(texts, model=EMBED_MODEL):
    """Embed a list of strings -> (n, d) float32 numpy array, order preserved."""
    if isinstance(texts, str):
        texts = [texts]
    resp = client.embeddings.create(model=model, input=texts)
    # sort by .index — do not assume the API returns them in order
    ordered = sorted(resp.data, key=lambda r: r.index)
    return np.array([r.embedding for r in ordered], dtype="float32")


t0 = time.time()
E = embed(TEXTS)
print(f"embedded {len(TEXTS)} chunks in {time.time() - t0:.2f}s")
print("matrix shape :", E.shape)          # (10, 1536)
print("dtype        :", E.dtype)
print("memory       :", E.nbytes / 1024, "KB")
print("vector norms :", np.round(np.linalg.norm(E, axis=1), 4))

embedded 10 chunks in 0.69s
matrix shape : (10, 1536)
dtype        : float32
memory       : 60.0 KB
vector norms : [0.9998 0.9998 0.9999 0.9999 0.9999 0.9999 1.     1.0001 1.0002 0.9999]


### 🔍 How the code works


- The `isinstance(texts, str)` guard means the function works for both a single query and a
  whole corpus. You will call it both ways.
- `sorted(resp.data, key=lambda r: r.index)` is defensive: the API returns an `index` field
  precisely so you can restore the input order. Getting this wrong silently mismatches every
  chunk with the wrong vector — a bug that is very hard to spot later.
- `dtype="float32"` halves the memory of the default float64 **and** matches what FAISS
  requires. Passing float64 to FAISS raises a type error.
- `np.linalg.norm(E, axis=1)` computes the length of each row. They all come back as
  **1.0**: `text-embedding-3` vectors arrive unit length. (If you ever shorten them with the
  `dimensions` parameter, or switch to a model that does not normalise, you must normalise
  them yourself — which is why every later step does it explicitly.)


### ⚙️ Behind the scenes


One HTTP request carried all ten texts. The server embedded them in parallel and returned
a JSON array; the client library parsed it into objects; `np.array(...)` copied the values
into one contiguous block of memory — 10 × 1536 × 4 bytes ≈ 60 KB.

That contiguity is the point. A search will multiply this whole block by one query vector
in a single BLAS call, which runs at hardware speed. A Python list of lists cannot do that.

**Batch limits.** In production you batch a few hundred texts per request and retry on
rate limits. The size that works depends on your tier — start at 100 and measure.


### 🧩 Where this sits in the RAG pipeline


Phase 1, step 3 (**Embed**), for the entire corpus. After this cell we have the vectors;
we just have nowhere sensible to put them yet. That is Step 5 onward.


### ❓ Questions the client is likely to ask

> **“What if one chunk is longer than the model's limit?”**  
> The request fails with an error. You catch it during chunking, not here — which is why chunk size is measured in tokens (with tiktoken) rather than characters.

> **“Why float32 rather than float64?”**  
> Half the memory, no measurable loss of retrieval accuracy, and it is what every ANN library expects. Some systems go further and quantise to 8 or even 1 bit per dimension.

> **“Does the order of DOCS matter?”**  
> Yes — row i of E must correspond to DOCS[i]. That positional link is our entire 'database' until Step 7. It is also the classic source of off-by-one bugs.

> **“How long would 100,000 chunks take?”**  
> Minutes, in batches, for a couple of dollars. It is a one-off cost — you only re-embed what changes.

### ✅ Checkpoint — pause here


Ask: **“What exactly is `E[3]`?”**

*Expected answer:* the 1536-number vector for `DOCS[3]`, the refund-policy chunk. If they
say “the fourth document”, push once more: it is the *vector for* that document, not the
document. The text still lives in `DOCS`.


---

## Step 3 · Cosine similarity, by hand

### 🎯 What we are going to implement


Write the cosine formula in NumPy and score a few pairs of chunks against each
other — no library, no index, just arithmetic.


### 💡 Why we are implementing it


Retrieval *is* this formula. If the client understands one line of maths today, make it
this one. Writing it by hand also demystifies every vector database afterwards: FAISS,
Chroma and Pinecone are all elaborate machinery for computing this number quickly.

In [5]:
def cosine(a, b):
    """Cosine similarity between two 1-D vectors: -1 (opposite) .. 1 (identical)."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


pairs = [(0, 1, "leave policy  vs  401k policy      (both HR)"),
         (0, 2, "leave policy  vs  remote work      (both HR)"),
         (0, 6, "leave policy  vs  API rate limit   (unrelated)"),
         (3, 4, "refunds       vs  support SLA      (both customer policy)"),
         (0, 0, "leave policy  vs  itself")]

for i, j, label in pairs:
    print(f"{cosine(E[i], E[j]):+.4f}   {label}")

+0.4037   leave policy  vs  401k policy      (both HR)
+0.3280   leave policy  vs  remote work      (both HR)
+0.0823   leave policy  vs  API rate limit   (unrelated)
+0.2746   refunds       vs  support SLA      (both customer policy)
+1.0000   leave policy  vs  itself


In [6]:
# Because OpenAI vectors are already unit length, the dot product alone
# gives the same number — and it is the form every fast index actually uses.
print("cosine     :", round(cosine(E[0], E[1]), 6))
print("dot product:", round(float(np.dot(E[0], E[1])), 6))

# The full 10x10 similarity matrix in one line — this is brute-force search
# for every query at once.
S = E @ E.T
print("\nsimilarity matrix shape:", S.shape)
print(np.round(S[:5, :5], 2))

cosine     : 0.403668
dot product: 0.403476

similarity matrix shape: (10, 10)
[[1.   0.4  0.33 0.18 0.3 ]
 [0.4  1.   0.28 0.13 0.31]
 [0.33 0.28 1.   0.22 0.3 ]
 [0.18 0.13 0.22 1.   0.27]
 [0.3  0.31 0.3  0.27 1.  ]]


### 🔍 How the code works


- `np.dot(a, b)` multiplies the two vectors element-wise and sums: Σ aᵢbᵢ.
- `np.linalg.norm(v)` is the vector's length, √(Σ vᵢ²).
- Dividing by both lengths removes magnitude from the comparison, leaving only the **angle**.
  That is the whole trick: two texts about the same topic point the same way regardless of
  how long they are.
- `E @ E.T` multiplies the (10 × 1536) matrix by its own transpose, producing a (10 × 10)
  matrix where `S[i, j]` is the similarity of chunk i to chunk j. The diagonal is 1.0.


### ⚙️ Behind the scenes


`np.dot` on float32 arrays calls into a BLAS kernel — the same optimised, vectorised code
that powers deep-learning frameworks. `E @ E.T` performs 10 × 10 × 1536 ≈ 154,000
multiply-adds and finishes instantly.

Now scale it in your head: 10 million chunks × 1536 dimensions is **15 billion** multiply-adds
*per query*, and the matrix alone occupies 61 GB of RAM. That single calculation is the
entire justification for ANN indexes in Step 6.

Expect real scores to look "high": with `text-embedding-3` models, genuinely unrelated
English sentences usually land around 0.05–0.25 rather than 0. What matters is the
**ranking**, not the absolute value.


### 🧩 Where this sits in the RAG pipeline


This is the arithmetic inside **Search**, the third box of Phase 2. Everything a vector
database adds is about not having to do all of it.


### ❓ Questions the client is likely to ask

> **“Why not Euclidean distance?”**  
> On unit-length vectors it ranks identically — d² = 2 − 2·cos(θ). Cosine is preferred because it is bounded to [−1, 1] and reads as a similarity rather than a distance.

> **“What is a 'good' score?”**  
> There is no universal threshold. It depends on the model and the corpus. Measure it: embed 20 known-good and 20 known-bad pairs and look at where they separate.

> **“Two chunks about leave scored 0.5 — is that broken?”**  
> No. 0.5 between two different HR topics is normal. Compare it with the 0.1 you get against an unrelated chunk — the gap is what retrieval uses.

> **“Can similarity be negative?”**  
> Mathematically yes, but with these models it almost never happens for natural text. Opposite meanings are not opposite directions — 'I love this' and 'I hate this' are close.

### ✅ Checkpoint — pause here


Ask: **“If I multiply every number in `E[0]` by 10, what happens to its cosine similarity
with `E[1]`?”**

*Expected answer:* nothing changes — dividing by the norms cancels the scale exactly.
Then ask what happens to the raw **dot product**: it becomes 10× larger. That is precisely
why normalising first is not optional when you use dot-product indexes.


---

## Step 4 · Brute-force similarity search

### 🎯 What we are going to implement


Write a `search(query, k)` function: embed the question, score it against every
chunk, return the top k with their text and metadata.


### 💡 Why we are implementing it


This is a complete, working retrieval system in eight lines. Building it before touching
FAISS means the client sees that a vector database is an **optimisation**, not a
requirement — and later, when FAISS returns the same ids, they will trust it.

In [7]:
def search(query, k=3):
    """Exact top-k search over the whole corpus. Returns (score, record) pairs."""
    q = embed(query)[0]                       # (1536,)
    q = q / np.linalg.norm(q)                 # be explicit, do not rely on the API
    scores = E @ q                            # (10,) — one dot product per chunk
    top = np.argsort(-scores)[:k]             # negate to sort descending
    return [(float(scores[i]), DOCS[i]) for i in top]


for score, doc in search("how much holiday am I entitled to?"):
    print(f"{score:.3f}  [{doc['id']}] {doc['text'][:70]}...")

0.403  [hr-1] New employees receive 15 days of paid annual leave in their first year...
0.286  [hr-3] Remote work is available up to three days per week with manager approv...
0.207  [fin-2] Expense reports must be filed within 60 days of the expense date....


In [8]:
# Try a few more, including one the corpus cannot answer.
for q in ["what happens if I want my money back?",
          "how fast does support reply for big customers?",
          "who won the world cup in 1998?"]:
    print(f"\nQ: {q}")
    for score, doc in search(q, k=2):
        print(f"   {score:.3f}  [{doc['id']}] {doc['text'][:60]}...")


Q: what happens if I want my money back?
   0.440  [pol-1] Refund requests must be submitted within 30 days of purchase...
   0.241  [fin-2] Expense reports must be filed within 60 days of the expense ...

Q: how fast does support reply for big customers?
   0.665  [pol-2] Enterprise customers receive dedicated support with a four-h...
   0.284  [pol-1] Refund requests must be submitted within 30 days of purchase...

Q: who won the world cup in 1998?
   0.055  [pol-2] Enterprise customers receive dedicated support with a four-h...
   0.047  [pol-1] Refund requests must be submitted within 30 days of purchase...


### 🔍 How the code works


- `embed(query)[0]` reuses the Step 2 function and takes the single row it returns.
- Normalising the query explicitly costs nothing and protects you the day you swap in an
  embedding model that does *not* return unit vectors.
- `E @ q` is a matrix–vector product: 10 dot products in one BLAS call, giving one score
  per chunk.
- `np.argsort(-scores)[:k]` sorts descending (NumPy only sorts ascending, hence the minus)
  and keeps the first k **indices** — which we then use to look up the original records.


### ⚙️ Behind the scenes


Two things happen per query: **one network round-trip** (~40–100 ms, and it dominates the
whole function) and **one matrix multiply** (~15,000 multiply-adds here, microseconds).

Watch the third query. “Who won the world cup in 1998?” still returns results — with lowish
scores, but it returns them. **Vector search always returns k results.** There is no
"no match" state. Handling that is not the retriever's job; it is the prompt's job, and we
do it in Step 9.


### 🧩 Where this sits in the RAG pipeline


Steps 1–3 of Phase 2 (**Embed the query → Search → top-k chunks**) are now complete and
working. Only prompt assembly and generation are missing.


### ❓ Questions the client is likely to ask

> **“Why does the irrelevant question still return chunks?”**  
> Because the search returns the nearest neighbours, and something is always nearest. You either threshold on the score or instruct the LLM to refuse when the context does not answer the question. The second is more reliable.

> **“What should k be?”**  
> Usually 3–8. Too small and you miss the answer; too large and you dilute the prompt with noise and pay for tokens you did not need.

> **“Can I search with a whole paragraph instead of a question?”**  
> Yes — and it often works better, because a longer query gives the embedding model more signal. That is the idea behind HyDE and query expansion in Session 71.

> **“Why re-embed the query every time? Can we cache it?”**  
> Absolutely — a dictionary keyed on the query string is a legitimate production optimisation for repeated questions.

### ✅ Checkpoint — pause here


Ask: **“We have a working search engine. What does FAISS give us that this doesn't?”**

*Expected answer:* speed at scale — and nothing else, at ten chunks. Then ask
**“at what corpus size would this start hurting?”** Somewhere around a million vectors on
one machine; the memory runs out before the CPU does.


---

## Step 5 · FAISS — an exact index

### 🎯 What we are going to implement


Put the same vectors into a FAISS `IndexFlatIP`, run the same query, confirm the
results are identical, then save the index to disk and load it back.


### 💡 Why we are implementing it


FAISS is the reference implementation of similarity search, and it is the engine hiding
inside many higher-level tools. Starting with the **exact** index proves the ideas transfer
unchanged: same vectors in, same neighbours out. The speed-up comes later — first, trust.

In [9]:
import faiss

d = E.shape[1]                       # 1536
index = faiss.IndexFlatIP(d)         # IP = inner product = cosine on unit vectors

Xb = E.copy()                        # copy: normalize_L2 works in place
faiss.normalize_L2(Xb)               # belt and braces — they are already unit length
index.add(Xb)                        # no training needed for a flat index

print("index type :", type(index).__name__)
print("is_trained :", index.is_trained)
print("ntotal     :", index.ntotal)
print("dimension  :", index.d)

index type : IndexFlatIP
is_trained : True
ntotal     : 10
dimension  : 1536


In [10]:
q = embed("how much holiday am I entitled to?")
faiss.normalize_L2(q)                       # shape (1, 1536) — FAISS wants 2-D

scores, ids = index.search(q, k=3)          # both are (n_queries, k) arrays

print("ids    :", ids[0])
print("scores :", np.round(scores[0], 4))
print()
for rank, (i, sc) in enumerate(zip(ids[0], scores[0]), start=1):
    print(f"{rank}. {sc:.3f}  [{DOCS[i]['id']}] {DOCS[i]['text'][:60]}...")

ids    : [0 2 9]
scores : [0.4029 0.2861 0.2069]

1. 0.403  [hr-1] New employees receive 15 days of paid annual leave in their ...
2. 0.286  [hr-3] Remote work is available up to three days per week with mana...
3. 0.207  [fin-2] Expense reports must be filed within 60 days of the expense ...


In [11]:
# Identical to our hand-written search? It must be — both are exact.
numpy_ids = [d["id"] for _, d in search("how much holiday am I entitled to?", k=3)]
faiss_ids = [DOCS[i]["id"] for i in ids[0]]
print("numpy :", numpy_ids)
print("faiss :", faiss_ids)
print("match :", numpy_ids == faiss_ids)

numpy : ['hr-1', 'hr-3', 'fin-2']
faiss : ['hr-1', 'hr-3', 'fin-2']
match : True


In [12]:
# Persistence: the index is just bytes on disk.
faiss.write_index(index, "handbook.index")
reloaded = faiss.read_index("handbook.index")
print("reloaded ntotal:", reloaded.ntotal)

# IMPORTANT: the text did not come along for the ride.
import os, pickle
with open("handbook_docs.pkl", "wb") as f:
    pickle.dump(DOCS, f)
print("index file :", os.path.getsize("handbook.index"), "bytes")
print("docs  file :", os.path.getsize("handbook_docs.pkl"), "bytes")

reloaded ntotal: 10
index file : 61485 bytes
docs  file : 1531 bytes


### 🔍 How the code works


- `IndexFlatIP(d)` creates a flat (exhaustive) index scored by **inner product**. On
  unit-length vectors that is exactly cosine similarity. `IndexFlatL2` is the Euclidean twin
  and, on normalised vectors, produces the same ranking with distances instead of scores.
- `faiss.normalize_L2(X)` normalises **in place** and returns `None`. Forgetting the `.copy()`
  would silently modify `E`.
- `index.add()` appends rows. `index.ntotal` is the running count; ids are assigned
  sequentially from 0 — they are *positions*, not your `hr-1` style ids.
- `index.search(q, k)` returns `(scores, ids)`, both shaped `(n_queries, k)`. You can pass
  many queries at once; that is much faster than looping.
- `write_index` / `read_index` handle persistence — of the vectors only.


### ⚙️ Behind the scenes


A flat index stores the raw matrix and, at query time, computes every dot product with SIMD
instructions and multiple threads. It is *the same arithmetic as Step 4*, hand-optimised —
typically several times faster than NumPy, and with a much better memory layout for large
batches.

Note what `index.search` gives back: **integers**. FAISS has no idea what a document is. The
mapping from row 0 to `hr-1` lives entirely in your `DOCS` list, which is why we pickled it
alongside. Lose that file and the index becomes meaningless numbers. This is the single
biggest practical difference between FAISS and a vector *database*.


### 🧩 Where this sits in the RAG pipeline


We have replaced the **Search** box with a real index. Phase 1's **Store** box is now
`write_index` + a pickle — crude, but functionally what Chroma will do properly in Step 7.


### ❓ Questions the client is likely to ask

> **“Why did `is_trained` print True with no training?”**  
> Flat indexes have nothing to learn — they keep every vector. IVF and PQ indexes must be trained on sample data first, and we will see that in Step 6.

> **“Can I delete a vector?”**  
> Not from a plain flat index — you need `IndexIDMap` and `remove_ids`, and IVF/HNSW have their own restrictions. Deletion being awkward is a real reason to prefer a database.

> **“Is FAISS faster than NumPy here?”**  
> At ten vectors, no — the difference is noise. At a million it is dramatic, and the gap widens further with IVF or HNSW.

> **“Does FAISS use the GPU?”**  
> There is a GPU build (`faiss-gpu`) that moves the index to VRAM. Rarely necessary below tens of millions of vectors.

### ✅ Checkpoint — pause here


Ask: **“I restart Python and load `handbook.index`. I search and get back id 7.
What do I know?”**

*Expected answer:* nothing useful without `handbook_docs.pkl` — id 7 is a row number. That
realisation is exactly the motivation for Step 7.


---

## Step 6 · FAISS — an approximate index (IVF)

### 🎯 What we are going to implement


Build a 20,000-vector index twice — once exact, once with IVF — and measure the
recall and the latency as we turn the `nprobe` dial.


### 💡 Why we are implementing it


Every vector database in production is making this trade-off on your behalf; hidden behind
Chroma's HNSW defaults and Pinecone's serverless magic is exactly this dial. Turning it by
hand, once, means the client will never again be confused about why a vector search
“missed” something. Use synthetic vectors: we are measuring the *index*, not the embeddings,
and 20,000 real embeddings would cost time and money for no extra insight.

In [13]:
# Synthetic vectors with cluster structure — like real embeddings, cheap to make.
rng = np.random.default_rng(0)
dim, n, n_clusters = 128, 20_000, 40

centres = rng.normal(size=(n_clusters, dim)).astype("float32") * 3
X = (centres[rng.integers(0, n_clusters, n)]
     + rng.normal(size=(n, dim)).astype("float32")).astype("float32")
faiss.normalize_L2(X)

print("corpus:", X.shape, f"{X.nbytes/1e6:.1f} MB")

corpus: (20000, 128) 10.2 MB


In [14]:
# The ground truth: an exact index.
flat = faiss.IndexFlatIP(dim)
flat.add(X)

queries = X[:200]                       # 200 queries, k=5
t0 = time.time(); _, truth = flat.search(queries, 5); t_flat = time.time() - t0
print(f"exact search: {t_flat*1000:.1f} ms for 200 queries")

exact search: 55.6 ms for 200 queries


In [15]:
# The approximate index: IVF. nlist = 4*sqrt(N) is the FAISS rule of thumb.
nlist = int(4 * np.sqrt(n))
quantiser = faiss.IndexFlatIP(dim)
ivf = faiss.IndexIVFFlat(quantiser, dim, nlist, faiss.METRIC_INNER_PRODUCT)

print("is_trained before:", ivf.is_trained)
ivf.train(X)                            # k-means over the corpus -> nlist centroids
ivf.add(X)
print("is_trained after :", ivf.is_trained, "| nlist =", nlist, "| ntotal =", ivf.ntotal)

is_trained before: False
is_trained after : True | nlist = 565 | ntotal = 20000


In [16]:
# The dial: how many of the nlist cells do we actually look inside?
print(f"{'nprobe':>7} {'recall@5':>10} {'latency':>10}   {'speed-up':>9}")
for nprobe in (1, 2, 4, 8, 16, 32):
    ivf.nprobe = nprobe
    t0 = time.time(); _, got = ivf.search(queries, 5); t = time.time() - t0
    recall = np.mean([len(set(g) & set(tr)) / 5 for g, tr in zip(got, truth)])
    print(f"{nprobe:>7} {recall:>10.3f} {t*1000:>8.1f} ms {t_flat/t:>8.0f}x")

 nprobe   recall@5    latency    speed-up
      1      0.420      1.9 ms       29x
      2      0.570      2.1 ms       26x
      4      0.759      2.9 ms       19x
      8      0.941      2.5 ms       23x
     16      1.000      2.5 ms       22x
     32      1.000      4.0 ms       14x


### 🔍 How the code works


- `IndexIVFFlat(quantiser, d, nlist, metric)` — the *quantiser* is a small flat index that
  holds the `nlist` cluster centroids. “Flat” at the end means the vectors inside each cell
  are stored uncompressed.
- `ivf.train(X)` runs k-means to find the centroids. This is why `is_trained` starts False:
  an IVF index must see representative data before it can accept any.
- `ivf.add(X)` assigns each vector to its nearest centroid — filing each document into a
  drawer.
- `ivf.nprobe` is how many drawers a query opens. `nprobe = nlist` reproduces exact search
  (and its cost); `nprobe = 1` is fastest and least accurate.
- **Recall@5** = of the 5 truly nearest neighbours, how many did we get back? That is the
  number you trade away for speed.


### ⚙️ Behind the scenes


The vector space is carved into `nlist` Voronoi cells. A query is compared against the
`nlist` centroids first (cheap), the nearest `nprobe` cells are chosen, and only the
vectors inside those cells are scored. With nlist = 565 and nprobe = 8 you touch roughly
8/565 ≈ 1.4% of the corpus.

Recall is lost at the **cell boundaries**: if the true nearest neighbour happens to sit in
the next cell over and that cell was not probed, it is invisible — the index will never
know it missed it. Raising `nprobe` widens the net.

Look at the table you just printed: recall climbs from about 0.4 to 1.0 while the search
stays an order of magnitude faster than exhaustive scanning. That curve is the whole story
of ANN search.

HNSW is the other family — a navigable graph rather than clusters. Its equivalent dial is
`efSearch`. Same trade-off, different mechanism; it is what Chroma uses in Step 7.


### 🧩 Where this sits in the RAG pipeline


Still the **Search** box — but now with an honest label: retrieval in production is
approximate, and a RAG system that “missed a document” has usually just met this trade-off
rather than a bug.


### ❓ Questions the client is likely to ask

> **“Isn't losing results dangerous?”**  
> It is a tunable risk. At nprobe = 16 here we lost nothing measurable and stayed ~20× faster. You choose the point on the curve — and you measure it, as we just did.

> **“Why does FAISS warn about too few training points?”**  
> k-means wants roughly 30–256 points per centroid. Fewer means poorly placed cells. Either lower nlist or train on more data.

> **“Should I always use IVF?”**  
> No. Below ~100k vectors, flat is simpler and exact. Reach for IVF or HNSW when latency or memory actually hurts.

> **“How do I pick nlist and nprobe for my data?”**  
> Start with nlist = 4·√N, then sweep nprobe exactly as above against your real queries and pick the lowest value that meets your recall target.

### ✅ Checkpoint — pause here


Ask: **“Your search misses a document you know is in the index. Name three possible causes.”**

*Expected answers:* (1) `nprobe` too low — an ANN miss; (2) the chunk's text does not
actually contain the answer in words the query resembles — a chunking problem; (3) query
and corpus were embedded with different models. Bonus: k was too small.


---

## Step 7 · ChromaDB — a real vector database

### 🎯 What we are going to implement


Store the same ten chunks in a persistent Chroma collection with their metadata,
query it, and then filter the search by category — something FAISS cannot do.


### 💡 Why we are implementing it


This is the step where the client sees what a database adds: text and metadata live
*with* the vectors, results come back as records rather than row numbers, the data
survives a restart, and you can narrow a search before it runs. For most projects Chroma
is where you should actually start.

In [17]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# PersistentClient writes to disk; chromadb.Client() would be in-memory only.
chroma = chromadb.PersistentClient(path="./chroma_store")

openai_ef = OpenAIEmbeddingFunction(model_name=EMBED_MODEL)   # reads OPENAI_API_KEY

collection = chroma.get_or_create_collection(
    name="handbook",
    embedding_function=openai_ef,
    configuration={"hnsw": {"space": "cosine"}},   # default is l2 — say what you mean
)
print("collection:", collection.name, "| count:", collection.count())

collection: handbook | count: 10


In [18]:
# Chroma calls the embedding model for us — we hand it text, not vectors.
collection.upsert(
    ids=[d["id"] for d in DOCS],
    documents=[d["text"] for d in DOCS],
    metadatas=[{"source": d["source"], "section": d["section"], "category": d["category"]}
               for d in DOCS],
)
print("count after upsert:", collection.count())

count after upsert: 10


In [19]:
res = collection.query(
    query_texts=["how much holiday am I entitled to?"],
    n_results=3,
)

for doc, meta, dist, _id in zip(res["documents"][0], res["metadatas"][0],
                                res["distances"][0], res["ids"][0]):
    print(f"distance {dist:.3f}  [{_id}] {meta['source']} §{meta['section']}")
    print(f"          {doc[:70]}...")

distance 0.597  [hr-1] handbook.pdf §4.2
          New employees receive 15 days of paid annual leave in their first year...
distance 0.714  [hr-3] handbook.pdf §2.1
          Remote work is available up to three days per week with manager approv...
distance 0.793  [fin-2] finance.md §5.9
          Expense reports must be filed within 60 days of the expense date....


In [20]:
# The feature FAISS does not have: filter first, then search.
res = collection.query(
    query_texts=["what approval do I need?"],
    n_results=3,
    where={"category": "finance"},          # only finance records are considered
)
for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
    print(f"[{meta['category']}] {doc[:70]}...")

print()
# And retrieval without any search at all — straight lookup by id.
print(collection.get(ids=["eng-2"])["documents"][0])

[finance] Vendor invoices above $10,000 require VP approval. Below that amount m...
[finance] Expense reports must be filed within 60 days of the expense date....

The API rate limit is 100 requests per minute on the Free tier and 1000 requests per minute on the Pro tier.


### 🔍 How the code works


- `PersistentClient(path=...)` creates a directory containing a SQLite database plus the
  HNSW index files. Re-run the notebook tomorrow and the data is still there.
- `OpenAIEmbeddingFunction` lets Chroma embed on your behalf — attached to the collection,
  so queries automatically use the same model as the documents. That whole class of bug
  disappears.
- `configuration={"hnsw": {"space": "cosine"}}` — **set this explicitly.** Chroma's default
  space is squared L2. On unit vectors the ranking is the same, but the numbers are not, and
  any threshold you tune will be wrong if you assumed cosine.
- `upsert` inserts or replaces by id, so re-running the cell is safe. (`add` would raise on
  duplicates.)
- `query()` returns parallel lists nested one level deep — index `[0]` is the first query,
  because you may pass several `query_texts` at once.
- `where={...}` filters on metadata; `get(ids=[...])` fetches records with no vector search.


### ⚙️ Behind the scenes


`upsert` did four things: called the embedding API for the ten documents, wrote ids, text
and metadata into SQLite, added the vectors to the HNSW index, and flushed to disk.

`query` reverses it: embed the query text, walk the HNSW graph for the nearest ids, fetch
those rows from SQLite, and return documents + metadata + distances together. Compare that
with Step 5, where we had to keep a pickle file to know what id 7 meant.

Chroma's single-node HNSW defaults are `ef_construction = 100`, `ef_search = 100`,
`max_neighbors = 16`. Those are the same knobs as Step 6's `nprobe`, on the graph family.
(Chroma Cloud uses a different index, SPANN, tuned for distributed scale.)

**Distance, not similarity.** With `space: cosine` Chroma returns `1 − cosine_similarity`.
Lower is better. Do not print it as a similarity score to end users without converting.


### 🧩 Where this sits in the RAG pipeline


This single object now covers **Store** in Phase 1 *and* **Search** in Phase 2 — plus the
metadata that makes citations and multi-tenant filtering possible.


### ❓ Questions the client is likely to ask

> **“Is Chroma a wrapper around FAISS?”**  
> No. It has its own HNSW implementation, plus SQLite for records and metadata. Same ideas, different code.

> **“Should I let Chroma call the embedding API?”**  
> For a prototype, yes — fewer moving parts and no model mismatch. For production you often want to embed in your own pipeline so you can batch, cache and retry.

> **“Why is the distance 0.4 when our cosine similarity was 0.6?”**  
> Because distance = 1 − similarity. Always check which one your database returns.

> **“Does the where filter run before or after the search?”**  
> Conceptually before — the search is restricted to matching records, so you always get n_results from within the filtered set, not leftovers from a global search.

> **“Can it handle a million documents?”**  
> On one node with enough RAM, yes. Beyond that you move to Chroma Cloud or a distributed store.

### ✅ Checkpoint — pause here


Ask: **“A customer asks us to delete all their data. Show me what you'd do — in FAISS, and
in Chroma.”**

*Expected answer:* Chroma — `collection.delete(where={"customer": "acme"})`, one call.
FAISS — rebuild the index from scratch, and separately fix your pickle. That contrast is the
argument for a database in one sentence.


---

## Step 8 · Pinecone — the managed option  *(optional)*

### 🎯 What we are going to implement


Create a serverless index, upsert our own vectors with metadata, and query it —
all guarded by an API-key check so the notebook still runs end to end without one.


### 💡 Why we are implementing it


Chroma is excellent until you need several machines, several tenants and an on-call
rotation. Pinecone shows the client what “someone else runs the database” looks like: the
same three operations (create, upsert, query) with none of the operational surface — and
with a few new concepts, notably namespaces and eventual consistency.

In [21]:
PINECONE_KEY = os.environ.get("PINECONE_API_KEY")
RUN_PINECONE = bool(PINECONE_KEY)
print("Pinecone section:", "ENABLED" if RUN_PINECONE else "SKIPPED (no PINECONE_API_KEY)")

Pinecone section: ENABLED


In [22]:
if RUN_PINECONE:
    from pinecone import Pinecone, ServerlessSpec

    pc = Pinecone(api_key=PINECONE_KEY)
    INDEX_NAME = "handbook-demo"

    # "Bring your own vectors": we embed with OpenAI, so dimension must match the model.
    if not pc.has_index(INDEX_NAME):
        pc.create_index(
            name=INDEX_NAME,
            vector_type="dense",
            dimension=1536,                 # text-embedding-3-small
            metric="cosine",
            spec=ServerlessSpec(cloud="aws", region="us-east-1"),   # free tier region
        )
        print("index created — this takes a few seconds")

    index_pc = pc.Index(INDEX_NAME)
    print(index_pc.describe_index_stats())

index created — this takes a few seconds
DescribeIndexStatsResponse(dimension=1536, total_vector_count=0, metric='cosine', namespaces=0)


In [23]:
if RUN_PINECONE:
    index_pc.upsert(
        namespace="handbook",
        vectors=[
            {"id": d["id"],
             "values": E[i].tolist(),                       # the vectors from Step 2
             "metadata": {"text": d["text"],                # store the text as metadata
                          "source": d["source"],
                          "section": d["section"],
                          "category": d["category"]}}
            for i, d in enumerate(DOCS)
        ],
    )
    print("upserted", len(DOCS), "vectors")
    time.sleep(5)        # Pinecone is eventually consistent — give it a moment
    print(index_pc.describe_index_stats())

upserted 10 vectors
DescribeIndexStatsResponse(dimension=1536, total_vector_count=10, metric='cosine', namespaces=1)


In [24]:
if RUN_PINECONE:
    qv = embed("how much holiday am I entitled to?")[0].tolist()

    res = index_pc.query(
        namespace="handbook",
        vector=qv,
        top_k=3,
        include_metadata=True,
        # filter={"category": {"$eq": "hr"}},     # uncomment to narrow the search
    )
    for m in res["matches"]:
        print(f"{m['score']:.3f}  [{m['id']}] {m['metadata']['text'][:60]}...")

0.403  [hr-1] New employees receive 15 days of paid annual leave in their ...
0.286  [hr-3] Remote work is available up to three days per week with mana...
0.207  [fin-2] Expense reports must be filed within 60 days of the expense ...


In [25]:
if RUN_PINECONE:
    # Clean up so the free-tier index does not sit around.
    # pc.delete_index(INDEX_NAME)
    print("remember to delete the index when the session is over")

remember to delete the index when the session is over


### 🔍 How the code works


- `Pinecone(api_key=...)` is the control-plane client; `pc.Index(name)` returns the
  data-plane handle you actually read and write through.
- `create_index(..., dimension=1536, metric="cosine", spec=ServerlessSpec(...))` is the
  bring-your-own-vectors path. The alternative, `create_index_for_model(...)`, lets Pinecone
  embed your text with one of its hosted models — convenient, but then the embedding model
  is theirs, not yours.
- **`dimension` must match your embedding model exactly.** 1536 for
  `text-embedding-3-small`. A mismatch is rejected at upsert time.
- Each vector is `{id, values, metadata}`. There is no separate “document” field — the text
  rides along inside `metadata` (up to 40 KB per record).
- `namespace="handbook"` partitions the index. Queries only see one namespace, which is how
  multi-tenant isolation is done.
- `filter={"category": {"$eq": "hr"}}` uses Pinecone's MongoDB-style operators:
  `$eq`, `$ne`, `$in`, `$gte`, `$and`, `$or` and friends.


### ⚙️ Behind the scenes


`create_index` provisions storage and compute in Pinecone's cloud; the index is not ready
the instant the call returns. `upsert` ships the vectors over HTTPS, where they are written
to object storage and picked up by an index builder.

That last part is why the `time.sleep(5)` is in the notebook: Pinecone is **eventually
consistent**. Freshly upserted records take a few seconds to become searchable. Query too
soon in a live demo and you get an empty result and an awkward pause.

The query itself runs on Pinecone's infrastructure — you never learn which index type it
used or how it was tuned. That is the trade: you buy scale and operations, you sell
visibility and control.


### 🧩 Where this sits in the RAG pipeline


Exactly the same **Store** and **Search** boxes as Chroma — moved off your machine. Nothing
about Phases 1 and 2 changes; only who operates them.


### ❓ Questions the client is likely to ask

> **“Why store the text in metadata? Isn't that wasteful?”**  
> It is how Pinecone works — there is no separate document field. Some teams instead store only an id in Pinecone and keep the text in Postgres or S3. Both are valid; storing it inline means one round-trip.

> **“What does a namespace actually do?”**  
> It partitions the index. Queries target one namespace, so tenant A can never see tenant B's data, and each query scans less.

> **“Is it expensive?”**  
> The Starter tier is free within limits; beyond that it is usage-priced on reads, writes and storage. Chroma self-hosted has no per-query cost but you run the machine.

> **“Can we use Pinecone's own embedding model instead?”**  
> Yes — create_index_for_model plus upsert_records, and Pinecone embeds your text with, for example, llama-text-embed-v2. Less code, less control over the vectors.

### ✅ Checkpoint — pause here


Ask: **“We are building a support bot for 50 customers, each with their own documents.
Sketch the storage.”**

*Expected answer:* one index, one namespace per customer (or a `customer_id` metadata field
plus a filter). Follow up: *“which is safer?”* → the namespace, because isolation is
structural rather than a filter someone might forget to pass.


---

## Step 9 · Close the loop — a minimal RAG answer

### 🎯 What we are going to implement


Take the retrieved chunks, build the prompt from Session 32a, call the chat model,
and print an answer with its sources.


### 💡 Why we are implementing it


Until now we have built half a system. This step joins retrieval to generation and makes
the whole point land: the model answers correctly about a handbook it has never seen, and
it can cite the section. It is also where the client sees how little code the “G” in RAG
actually is.

In [26]:
SYSTEM = """You are a helpful assistant for company employees.
Answer ONLY from the context provided below.
If the context does not contain the answer, say exactly:
"I could not find that in the company documents."
Cite the source and section for every fact you use."""


def build_prompt(question, retrieved):
    context = "\n\n".join(
        f"[{d['source']} §{d['section']}] {d['text']}" for _, d in retrieved
    )
    return f"CONTEXT:\n{context}\n\nQUESTION: {question}"


def rag_answer(question, k=3, model="gpt-4o-mini"):
    retrieved = search(question, k=k)                 # our Step 4 function
    prompt = build_prompt(question, retrieved)
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content, retrieved

In [27]:
for question in ["How many days of leave do I get in my first year?",
                 "Can I get a refund after 45 days?",
                 "What is the company dress code?"]:
    answer, used = rag_answer(question)
    print("Q:", question)
    print("A:", answer)
    print("   sources:", [f"{d['source']} §{d['section']}" for _, d in used])
    print("-" * 78)

Q: How many days of leave do I get in my first year?
A: You receive 15 days of paid annual leave in your first year. [handbook.pdf §4.2]
   sources: ['handbook.pdf §4.2', 'handbook.pdf §2.1', 'handbook.pdf §4.7']
------------------------------------------------------------------------------
Q: Can I get a refund after 45 days?
A: I could not find that in the company documents.
   sources: ['policies.md §1.1', 'finance.md §5.9', 'policies.md §1.4']
------------------------------------------------------------------------------
Q: What is the company dress code?
A: I could not find that in the company documents.
   sources: ['handbook.pdf §4.7', 'security.md §6.0', 'handbook.pdf §2.1']
------------------------------------------------------------------------------


### 🔍 How the code works


- `SYSTEM` does three jobs: restrict the model to the context, give it an explicit escape
  hatch, and demand citations. All three are load-bearing.
- `build_prompt` labels every chunk with its source *inside the text*. The model cannot cite
  metadata it cannot see — this is how a citation becomes possible at all.
- `rag_answer` is the entire pipeline in three lines: retrieve → assemble → generate.
- `temperature=0` makes the answer as reproducible as the API allows. For grounded Q&A you
  almost never want creativity.


### ⚙️ Behind the scenes


Two API calls per question: one to the embedding model (~15 tokens) and one to the chat
model (~600 input tokens, a few dozen out). Well under a cent.

Watch the third question. Nothing in the corpus mentions a dress code, but retrieval still
returned three chunks — remember, it always does. The refusal comes entirely from the
system prompt. **Retrieval finds; the prompt decides what counts as an answer.**

Watch the second one too: the corpus says refunds are limited to 30 days, and the question
asks about 45. A grounded model reasons from the retrieved rule rather than inventing a
policy. That is the behaviour you are selling.


### 🧩 Where this sits in the RAG pipeline


This completes Phase 2: **question → embed → search → prompt → answer**, with sources.
Everything in Sessions 33–38 is a refinement of this loop — better chunking, re-ranking,
evaluation, multimodal inputs, agents that decide when to search.


### ❓ Questions the client is likely to ask

> **“Why did it refuse on the dress code but still retrieve chunks?”**  
> Because retrieval and generation are separate. The retriever has no notion of relevance thresholds; the prompt supplies the judgement.

> **“What if the model ignores the instruction and answers anyway?”**  
> It happens. Mitigations: a stronger system prompt, a relevance threshold on the scores, or a second model call that checks the answer against the context. That is Session 73, RAG evaluation.

> **“Should the chunks go in the system message or the user message?”**  
> Either works. Context in the user message is the common pattern because it varies per question and the system message can then be cached.

> **“How do I stop the prompt from getting too long?”**  
> Cap k, cap chunk size, and — later — compress or re-rank the retrieved context before it goes in.

### ✅ Checkpoint — pause here


Ask: **“Change one thing so this system can answer questions about your own PDFs.”**

*Expected answer:* replace the corpus — load, chunk and embed the PDFs. Nothing else in the
notebook changes. If they say that unprompted, they have understood RAG.


---

## Step 10 · Recap, comparison and exercises

### What we built

```
DOCS  ──embed──►  E (10 × 1536)  ──►  cosine / dot product  ──►  top-k  ──►  prompt  ──►  answer
                       │
                       ├──►  FAISS IndexFlatIP      exact, in RAM, ids only
                       ├──►  FAISS IndexIVFFlat     approximate, tunable via nprobe
                       ├──►  Chroma collection      vectors + text + metadata, on disk
                       └──►  Pinecone index         the same, run by somebody else
```

### The three tools, one more time

| | FAISS | ChromaDB | Pinecone |
|---|---|---|---|
| What it is | A library | An embedded database | A managed service |
| Stores the text | No — ids only | Yes | Yes (in metadata) |
| Metadata filters | No | Yes | Yes |
| Persistence | `write_index` | Automatic | Managed |
| Index control | Total | HNSW, tunable | Hidden |
| Scale ceiling | One machine's RAM | One node (or Cloud) | Effectively none |
| Reach for it when | You need speed or want to see the internals | You are building an app | You are running a product |

### Five things worth remembering

1. An embedding is a fixed-length vector whose **direction** encodes meaning.
2. **Index and query must use the same embedding model.** Always.
3. Normalise, then use the dot product — that is cosine similarity at hardware speed.
4. ANN indexes trade a measurable amount of recall for a large amount of speed. You choose
   the point on that curve; you do not have to accept a default.
5. A vector database is an index **plus** storage, metadata and operations. The metadata is
   what turns a demo into a product.

---

### Exercises for the client

1. **Threshold hunting.** Run 10 questions you know the corpus answers and 10 it does not.
   Plot the top score for each. Where would you set a cut-off — and how much do you regret it?
2. **Break the model match.** Embed the corpus with `text-embedding-3-small` and the query
   with `text-embedding-3-large`. Predict the result before running it, then explain what you see.
3. **Chunking.** Merge all ten records into one giant document, embed that, and search again.
   Why do the results get worse?
4. **Metadata design.** Add an `effective_date` to each record and write a query that only
   searches policies in force after 2025.
5. **Rebuild Step 4 with `IndexFlatL2`.** Confirm the ranking is identical and explain why,
   using d² = 2 − 2·cos(θ).

### What comes next

- **Session 33 — RAG Pipeline Lab I:** real documents, real loaders, real chunking strategies.
- **Session 34 — RAG Pipeline Lab II:** the full application, wired end to end.
- **Session 35 — Advanced RAG:** hybrid search, query rewriting, re-ranking — every one of
  them a fix for a failure mode you met today.

---

*Session 32b · Computer Vision & AI series · Phase 9 — RAG Systems*